# Step 3: Baseline Models
Evaluates two baselines against the simulated candidate pairs:
- **Rule-Based**: hard threshold cutoffs (no learning)
- **Logistic Regression**: linear classifier via scikit-learn

**To run:** Use `Kernel → Restart & Run All` to execute all cells in order.

In [1]:
# Run from project root
import os
if os.path.basename(os.getcwd()) in ['models', 'simulation', 'data']:
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')
print('Working directory:', os.getcwd())

Working directory: /Users/krunal/Desktop/final-project-team-tle


In [2]:
import pandas as pd
import yaml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

class RuleBasedBaseline:
    """
    Simple threshold baseline. Accepts any pair where detour < max_km
    and time difference < max_mins. No learning, just hard cutoffs.
    """
    def __init__(self, max_km=3.0, max_mins=15.0):
        self.max_km = max_km
        self.max_mins = max_mins
        
    def predict(self, df):
        """Predict 1 (accept) or 0 (reject) based on threshold cutoffs."""
        predictions = (
            (df['detour_distance_km'] <= self.max_km) & 
            (df['time_diff_mins'] <= self.max_mins)
        ).astype(int)
        return predictions
    
    def evaluate(self, y_true, y_pred):
        """Exposes raw viability metrics for comparisons."""
        return {
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred, zero_division=0),
            'F1': f1_score(y_true, y_pred, zero_division=0)
        }

class LogisticRegressionBaseline:
    """
    Logistic regression baseline with standardized features.
    Provides a linear decision boundary to compare against XGBoost.
    """
    def __init__(self):
        # Balanced prevents class-imbalance from zero-crushing predictions
        self.model = LogisticRegression(class_weight='balanced')
        # Scaling is required for distance math, otherwise KM vastly outweighs Ratio logic
        self.scaler = StandardScaler()
        # Explicit feature definition
        self.features = ['driver_orig_dist', 'detour_distance_km', 'overlap_ratio', 'time_diff_mins', 'driver_approx_fare', 'rider_approx_fare']
        
    def prepare_data(self, df):
        X = df[self.features]
        y = df['label'] 
        return X, y
        
    def train_and_evaluate(self, df):
        """Pipelines splitting, scaling, and scoring into one traceable execution layer."""
        X, y = self.prepare_data(df)
        
        # 80/20 train test split locked randomly for reproducibility
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # Prevent Data Leakage by fitting standard scaler exclusively on training subset
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        self.model.fit(X_train_scaled, y_train)
        y_pred = self.model.predict(X_test_scaled)
        
        metrics = {
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1': f1_score(y_test, y_pred, zero_division=0)
        }
        
        return metrics

In [3]:
# Load config and data
with open("config/dev_config.yaml", "r") as f:
    config = yaml.safe_load(f)
sim = config["simulation"]

print("Loading simulated pairs...")
df = pd.read_parquet("data/simulated/candidate_pairs.parquet")
print(f"Loaded {len(df)} pairs. Label distribution:\n{df['label'].value_counts()}")

Loading simulated pairs...
Loaded 750 pairs. Label distribution:
label
0    425
1    325
Name: count, dtype: int64


In [4]:
print("--- Rule-Based Baseline ---")
rb = RuleBasedBaseline(
    max_km=sim["max_detour_km"],
    max_mins=sim["max_detour_mins"]
)
y_pred_rb = rb.predict(df)
rb_metrics = rb.evaluate(df['label'], y_pred_rb)
print(rb_metrics)

--- Rule-Based Baseline ---
{'Accuracy': 0.9333333333333333, 'Precision': 0.8666666666666667, 'Recall': 1.0, 'F1': 0.9285714285714286}


In [5]:
print("--- Logistic Regression Baseline ---")
lr = LogisticRegressionBaseline()
lr_metrics = lr.train_and_evaluate(df)
print(lr_metrics)

--- Logistic Regression Baseline ---
{'Accuracy': 0.9, 'Precision': 0.8375, 'Recall': 0.9710144927536232, 'F1': 0.8993288590604027}
